# Intro to Financial Computation

Objective is to introduce:
- Time-series data
- Computation with NumPy
- Filtering

In [ ]:
# Run this cell first to install required packages
try:
    # Check if we're running in Colab
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing required packages...")
    !pip install -q numpy matplotlib requests
except ImportError:
    IN_COLAB = False
    print("Running in local Jupyter environment.")

import numpy as np
import matplotlib.pyplot as plt
print(" Setup complete!")

Running in Google Colab. Installing required packages...
 Setup complete!


# Introduction

In this notebook, we will explore basic financial computations using NumPy.  
Our goals are:
- Review descriptive statistics on stock price data.  
- Learn how moving averages are computed and applied.  
- Compare **Simple Moving Average (SMA)**, **Weighted Moving Average (WMA)**, and **Volume Weighted Average Price (VWAP)**.  

By the end, you will see how these indicators are used to smooth noisy data and make decisions in trading contexts.

# Financial Data Source

Based on the book `Numpy for Beginners`: https://learning.oreilly.com/library/view/numpy-beginners/9781785281969/

This notebook uses 5 years of financial data, up to Feb 2018, from the S&P 500 dataset: https://www.kaggle.com/camnugent/sandp500/data

The data is hosted on Google Drive and will be automatically downloaded when you run the notebook (both in Colab and regular Jupyter notebooks).

> Stock market data can be interesting to analyze and as a further incentive, strong predictive models can have large financial payoff. The amount of financial data on the web is seemingly endless. A large and well structured dataset on a wide array of companies can be hard to come by. Here I provide a dataset with historical stock prices (last 5 years) for all companies currently found on the S&P 500 index.

Google Drive link to data.csv: https://drive.google.com/file/d/1jG7Ml-hZp3LwYrbDe3asgiyPI5oGM6n4/view?usp=drive_link

Cols in data.csv:
- Stock name
- Date
- 'blank'
- Open price
- Daily High
- Daily Low
- Closing Price
- Trade Volume

We will focus on the closing price and trade volume for each day.

In [ ]:
import numpy as np
import os

# Function to load data from Google Drive or local directory
def load_data(file_id='1jG7Ml-hZp3LwYrbDe3asgiyPI5oGM6n4', filename='data.csv'):
    """
    Load financial data from Google Drive (works in both Colab and regular notebooks)

    Parameters:
    -----------
    file_id : str
        Google Drive file ID (from the shared link)
    filename : str
        Filename to save locally

    Returns:
    --------
    tuple
        (closing_prices, volumes) as numpy arrays
    """
    try:
        # First check if file already exists locally
        if os.path.exists(filename):
            print(f"Loading {filename} from local directory...")
            return np.loadtxt(filename, delimiter=',', usecols=(6,7), unpack=True)

        # Check if running in Colab
        in_colab = 'google.colab' in str(get_ipython())

        if in_colab:
            # Use gdown in Colab (simpler approach)
            print(f"Downloading {filename} from Google Drive...")
            !pip install -q gdown
            !gdown {file_id} -O {filename}
        else:
            # For regular Jupyter notebooks, download directly using requests
            print(f"Downloading {filename} from Google Drive...")
            import requests

            # Create the download URL from the file ID
            url = f"https://drive.google.com/uc?id={file_id}"

            # Download the file
            response = requests.get(url)
            with open(filename, 'wb') as f:
                f.write(response.content)

        # Load and return the data
        print(f"Successfully downloaded {filename}")
        return np.loadtxt(filename, delimiter=',', usecols=(6,7), unpack=True)

    except Exception as e:
        print(f"Error loading data: {e}")

        # Fallback to original path if download fails
        fallback_path = '/home/memo/public/eaix/'
        if os.path.exists(fallback_path + filename):
            print(f"Falling back to local path: {fallback_path}")
            return np.loadtxt(fallback_path + filename, delimiter=',', usecols=(6,7), unpack=True)

        # If all else fails, raise the error
        raise FileNotFoundError(f"Could not load {filename}. Please check your internet connection or file path.")

# Load the main data.csv file
# The file_id comes from your shared Google Drive link
c, v = load_data(file_id='1jG7Ml-hZp3LwYrbDe3asgiyPI5oGM6n4', filename='data.csv')

Loading data.csv from local directory...


links:
- data.csv: https://drive.google.com/file/d/1jG7Ml-hZp3LwYrbDe3asgiyPI5oGM6n4/view?usp=drive_link
- AAPL_data.csv: https://drive.google.com/file/d/1uBUC0aQSmTPdHFJ05fv6aZGu2MGw3EAW/view?usp=drive_link
- AMZN_data.csvL https://drive.google.com/file/d/1fnjsjVdBqHRPHjMNST6i-bY2TFA7_iZM/view?usp=drive_link
- MSFT_data.csv: https://drive.google.com/file/d/1314rW_N3np8Mh8g74TPwSUBfI1I7lfA9/view?usp=drive_link

## Exercise 1: Exploring the Dataset

How much data is in data.csv?

In [ ]:
# Solution: Check the length of the data arrays
print(f'{len(c)=}, {len(v)=}, {c.shape=}, {c[:5]=}')
# You can also check the shape of the arrays

# Display the first few data points


len(c)=30, len(v)=30, c.shape=(30,), c[:5]=array([336.1 , 339.32, 345.03, 344.32, 343.44])


## Ex:

Read one of the of csv of your chossing for 5 years:
AAPL_data.csv, AMZN_data.csv or MSFT_data.csv

Save closing and trade volume into 2 arrays: `cp` and `tv`

Note: Col order is same but the stock name is moved to the the end

In [ ]:
# Example: Load AAPL stock data
stock_symbol = 'AAPL'  # Change to 'MSFT' or 'AMZN' as desired
filename = f"{stock_symbol}_data.csv"

# Add the file IDs for different stock files
stock_file_ids = {
    'AAPL': 'xxx',  # Replace with actual file ID for AAPL
    'MSFT': 'xxx',  # Replace with actual file ID for MSFT
    'AMZN': 'xxx'   # Replace with actual file ID for AMZN
}

# Get the file ID for the selected stock (if it exists)
file_id = stock_file_ids.get(stock_symbol)

if file_id:
    # Load closing prices and volume for the selected stock
    cp, tv = load_data(file_id=file_id, filename=filename)

    # Print first few values to verify
    print(f"First 5 closing prices for {stock_symbol}:")
    print(cp[:5])
    print(f"\nFirst 5 trading volumes for {stock_symbol}:")
    print(tv[:5])
else:
    print(f"No file ID defined for {stock_symbol}. Please add it to the stock_file_ids dictionary.")

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=xxx

but Gdown can't. Please check connections and permissions.
Successfully downloaded AAPL_data.csv
Error loading data: AAPL_data.csv not found.


FileNotFoundError: Could not load AAPL_data.csv. Please check your internet connection or file path.

# Range

Calculate the **range of the closing prices** using np.ptp() (peak-to-peak)

In [ ]:
np.ptp(c)

In [ ]:
#Ex: Calculate your own p2p w/o loops:


## Basic Statistics

Before we build financial indicators, let’s recall some descriptive statistics.  
We’ll use stock **closing prices** as an example.  

In [ ]:
# TODO: Compute basic statistics of column 'Close'
# Use NumPy functions np.mean, np.median, np.std
# Example: np.mean(c)

mean_close = ...
median_close = ...
std_close = ...

print("Mean:", mean_close)
print("Median:", median_close)
print("Std Dev:", std_close)

## Calculating financial indicators


### Volume Weighted Average Price (VWAP)

VWAP uses **both price and volume** to reflect the average trading price across the day.  
It is often used as a benchmark to measure execution quality.

See video at:
https://www.investopedia.com/terms/v/vwap.asp

Formula from: https://en.wikipedia.org/wiki/Volume-weighted_average_price

$$ P_{\mathrm{VWAP}} = \frac{\sum_{j}{P_j \cdot Q_j}}{\sum_j{Q_j}} \,$$
where:

- $ P_{\mathrm{VWAP}}$ is Volume Weighted Average Price;
- $P_j$ is price of trade $j$;
- $Q_j$ is quantity of trade $j$;
- $j$ is each individual trade that takes place over the defined period of time

**Note:** This is usually a single-day indicator. Our monthly analysis is probably an abuse.

## Ex:
How is it calculated for the example in video?

In [ ]:
# NumPy approach:
vwap = np.average(c, weights=v)
print("VWAP =", vwap)

## Ex:

- How do we view the docstring?

- If we don't use weights, i.e. w=1, what do we get?

## Ex:
Write your own VWAP w/o `for` loops, using the math formula. Utilize np methods

### TWAP

Time-weighted average price

https://en.wikipedia.org/wiki/Time-weighted_average_price

**Note:** This is usually a single-day indicator. Our montly analysis is probably an abuse.

> The most common use of TWAP is for distributing big orders throughout the trading day. For example let’s say you want to buy 100,000 shares of Morgan Stanley. Putting one such a big order would vastly impact the market and the price most likely would start to raise. To prevent that, investor can define time period in TWAP Strategy over which they want to buy shares. It will slice evenly big order into smaller ones and execute them over defined period.

https://empirica.io/blog/twap-strategy/

In [ ]:
t = np.arange(len(c))
print("twap =", np.average(c, weights=t))

In [ ]:
# Ex?
# repeat process over the open price or MSFT price?

### Returns

A lot of different return calculations over various time periods exist.

Mainly to calculate profit.

#### Basic Return
(Basic) Return is calculated as the price difference over by the previous price:

$$r_i = (c_{i+1} - c_i) \,/\, c_i$$

If $i=1:$
$$r_1 = (c_2 - c_1) \,/\, c_1$$

Note: $i$ can be over a variety of range. we will go daily.

If our data is 30 days long, how many daily returns do we get?

What are the units for $r\,$?

## Ex:
np implementation of returns?



In [ ]:
#Hint: See np.diff()


## Ex:

Which are the postive returns?

**Hint**: Use masking

#### Log Return

Logarithmic return is the compund interest return.

Formulated as:

$$ r_l = \mathrm{log}_e \left(\frac{V_f}{V_i}\right) $$


https://en.wikipedia.org/wiki/Rate_of_return#Logarithmic

https://en.wikipedia.org/wiki/Rate_of_return#Comparing_ordinary_return_with_logarithmic_return


## Ex:

Calculate the log returns for each day in data?

**Hint**: How can we re-write the log ratio?

Transformations are your friend!!!

From here, we can easily calculate **volatility** to better understand the financial securities.

If we don't have access to log returns, volatility can also be estimated using **Taylor** series.

# Ex

Best days to sell/buy:

Which day, on avg, has the highest closing price?

Which is the cheapest?

**Note**: Python has a datetime to help us find the days fo the weeks based on the stock date, but, we can cheat. Look at the data.csv and map M-F.
In this case, 2/21 is a holiday and our dates get shifted.

## Simple Moving Average (SMA)

We will now look at this data over **a moving period**, called a **window**.

Common practice for **time-series** data.


A **Simple Moving Average** smooths out short-term fluctuations by taking the mean over a rolling window.  
This is one of the most widely used indicators in finance.  

> A simple moving average (SMA) calculates the average of a selected range of prices, usually closing prices, by the number of periods in that range.

https://www.investopedia.com/terms/s/sma.asp

Explain how a window works:

N = 5:


| Days   |      SMA     |
|----------|:-------------:|
| 0-4 |  avg(c[0-4]) |
| 1-5 |  avg(c[0-4]) |
| 2-6 | avg(c[2-6]) |
| ... | ... |
| 25-29 | avg(c[25-29]) |
    

## Ex:

How many SMA results do we have for N=5 from data.csv?

Implement sma using a single `for` loop?

In [ ]:
#desired:
# [341.642 343.722 346.234 348.268 351.036 353.256 355.326 356.786 357.726
#  358.72  359.472 358.214 354.1   350.644 346.594 344.566 345.096 347.236
#  349.136 352.472 354.84  355.27  356.56  356.63  354.052 352.45 ]

N =5

#Note: sma=[], sma.append() is quicker to write but a list.
# Below should be qicker to run np b/c memory is pre-allocated.
smafoor=np.zeros([len(c)-N+1])

Note that during the each iteration, we grab a **window**, length of *N* and calculate its average.

You could place the `mean()` with another `for` loop, but the vectorization helps readability and speed in this case!



What does the SMA look like compared to the closing price?


In [ ]:
import matplotlib.pyplot as plt

sma = smafoor
t = np.arange(N - 1, len(c))
plt.plot(t, c[N-1:], lw=1.0, label="Data")
plt.plot(t, sma, '--', lw=2.0, label="Moving average")
plt.title("5 Day Moving Average")
plt.xlabel("Days")
plt.ylabel("Price ($)")
plt.grid()
plt.legend()
plt.show()

Change N from 5 to 1 and 10.

What do you observe?

Does this remind you of something?

### Online SMA

The above algorithm works great for a window size of 5. But what if N is 100? or 10M? What if $x$ is m-dimensional?

There is a version alternative to the brute-force method defined above!

Derive online algorithm here.

$$S_{n+1} = S_n + \frac{x_{n+1}-S_n}{n+1} $$

In [ ]:
## Your code for online SMA goes here.


## Trading Strategies Using Moving Averages

Moving averages are fundamental tools in technical analysis for developing trading strategies. Here are some common strategies:

### 1. Moving Average Crossover [link investopedia](https://www.investopedia.com/terms/g/goldencross.asp#toc-golden-cross-vs-death-cross)
- **Golden Cross**: When a short-term SMA crosses above a long-term SMA (e.g., 50-day crossing above 200-day)
  - This is considered a bullish signal
- **Death Cross**: When a short-term SMA crosses below a long-term SMA
  - This is considered a bearish signal

### 2. Price and Moving Average Crossover
- Buy when price crosses above a moving average
- Sell when price crosses below a moving average

### 3. Moving Average Ribbon [link trendspider](https://trendspider.com/learning-center/what-is-a-moving-average-ribbon/)
- Plot multiple moving averages with different timeframes
- When they fan out or converge, it can signal trend strength or weakness

### TODO: implement a simple moving average crossover strategy

## From Financial Indicators to Algorithmic Trading

The financial indicators we've studied in this notebook form the foundation of algorithmic trading systems. Here's how they fit into the bigger picture:


### The Algorithmic Trading Pipeline:

1. **Data Collection**
   - Market data feeds (prices, volumes)
   - Alternative data (news, social media)
   - Economic indicators

2. **Signal Generation**
   - Technical indicators (like our SMAs, VWAP)
   - Statistical models (regression, time series)
   - Machine learning models

3. **Risk Management**
   - Position sizing
   - Stop-loss orders
   - Portfolio diversification

4. **Execution**
   - Order types (market, limit)
   - Execution algorithms (TWAP, VWAP)
   - Minimizing market impact



### Why Computer Science Skills Matter in Finance:

- **Data Processing**: Working with large volumes of market data
- **Algorithm Design**: Creating efficient trading strategies
- **Optimization**: Minimizing transaction costs and maximizing returns
- **Backtesting**: Testing strategies against historical data
- **Real-time Systems**: Executing trades with low latency

In your future careers, the NumPy skills you're learning today could be applied to build sophisticated trading systems that process millions of data points and execute trades in microseconds.

We looked at how to efficiently calculate many financial markers with NumPy for a time-series dataset.

Where can we go form here?

- SMA: Another convoluted approach
- Weighted / Exponential Moving Average
- Smoothing with fancy windows (Blackmann, Hamming, Hanning, etc.)
- Finding a meaningful relationship: Covariance / Correlation
- Frequnecy analysis?


# Conclusion

- **SMA** is easy to compute and useful for smoothing, but gives equal weight to all periods.  
- **WMA** emphasizes more recent data, making it more responsive to trends.  
- **VWAP** incorporates both price and trading volume, and is widely used by traders to benchmark execution.  

In practice, traders often compare these indicators to detect momentum, trend reversals, or trading opportunities.  

In [ ]:
#The end